# 04 — Forecasting protocol and artifact audit

The backend pipeline is the canonical implementation. Run `python src/backend/main.py` from the repository root to regenerate artifacts; this notebook is read-only and does not train, select, or persist models.

The frozen protocol has three disjoint windows:

- Model selection: **2025-01-01 through 2025-05-30**. Optuna and candidate comparison are allowed only here.
- Policy calibration: **2025-05-31 through 2025-09-30**. Frozen methods generate out-of-selection residuals; no retuning occurs here.
- Final evaluation: **2025-10-01 through 2025-12-31**. This holdout is untouched until every choice is frozen.

All forecasts follow a daily rolling-one-step information contract: the row for day `t` may use actual demand observed through `t-1` only.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "backend").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "backend").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

EXPECTED_WINDOWS = {
    "model_selection": (pd.Timestamp("2025-01-01"), pd.Timestamp("2025-05-30")),
    "policy_calibration": (pd.Timestamp("2025-05-31"), pd.Timestamp("2025-09-30")),
    "final_evaluation": (pd.Timestamp("2025-10-01"), pd.Timestamp("2025-12-31")),
}
pd.DataFrame(
    [{"window": name, "start": bounds[0], "end": bounds[1]} for name, bounds in EXPECTED_WINDOWS.items()]
)

## Causal feature contract

The cell below mirrors `src/backend/features/feature_engineering.py` for regression parity. It is retained as an executable teaching reference; production execution imports the backend function.

In [ ]:
def create_features(frame):
    """Create the canonical causal feature schema used by the backend."""
    df = frame.copy().sort_values(["sku_id", "date"]).reset_index(drop=True)

    # Calendar features are known at forecast time.
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["year"] = df["date"].dt.year
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_month_start"] = (df["day_of_month"] <= 5).astype(int)
    df["is_month_end"] = (df["day_of_month"] >= 25).astype(int)

    # Cyclical calendar features.
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    grouped_target = df.groupby("sku_id", observed=True)["demand"]
    for lag in [1, 7, 14, 30, 60, 90]:
        df[f"lag_{lag}"] = grouped_target.shift(lag)

    for window in [30, 60, 90]:
        history = grouped_target.transform(
            lambda series: series.shift(1).rolling(window, min_periods=window).mean()
        )
        std = grouped_target.transform(
            lambda series: series.shift(1).rolling(window, min_periods=window).std()
        )
        df[f"rolling_mean_{window}"] = history
        df[f"rolling_std_{window}"] = std
        df[f"rolling_cv_{window}"] = std / (history + 1e-6)

    df["rolling_diff_30_60"] = df["rolling_mean_30"] - df["rolling_mean_60"]
    df["rolling_diff_30_90"] = df["rolling_mean_30"] - df["rolling_mean_90"]
    df["rolling_diff_60_90"] = df["rolling_mean_60"] - df["rolling_mean_90"]
    df["rolling_growth_30_60"] = df["rolling_mean_30"] / (df["rolling_mean_60"] + 1e-6)
    df["rolling_growth_30_90"] = df["rolling_mean_30"] / (df["rolling_mean_90"] + 1e-6)
    df["rolling_growth_60_90"] = df["rolling_mean_60"] / (df["rolling_mean_90"] + 1e-6)

    for alpha in [0.3, 0.5, 0.7]:
        alpha_name = str(alpha).replace(".", "")
        df[f"ewm_a{alpha_name}"] = grouped_target.transform(
            lambda series: series.shift(1).ewm(alpha=alpha, adjust=False).mean()
        )

    df["month_index"] = (df["year"] - df["year"].min()) * 12 + df["month"]
    return df

## Canonical artifact audit

`best_params.json` records the frozen protocol and feature schema. `validation_forecast.csv` contains policy-calibration forecasts only, while `forecast.csv` contains the locked final evaluation. `forecast_metrics.csv` is the canonical hierarchy of WAPE, bias, MASE, and RMSSE.

In [ ]:
artifact_paths = {
    "params": ROOT / "params" / "best_params.json",
    "calibration": ROOT / "data" / "processed" / "validation_forecast.csv",
    "final_forecast": ROOT / "data" / "processed" / "forecast.csv",
    "forecast_metrics": ROOT / "data" / "processed" / "forecast_metrics.csv",
    "feature_importance": ROOT / "params" / "feature_importance.csv",
}
missing = [str(path.relative_to(ROOT)) for path in artifact_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing canonical artifacts. Run `python src/backend/main.py`: " + ", ".join(missing)
    )

params = json.loads(artifact_paths["params"].read_text(encoding="utf-8"))
calibration_forecast = pd.read_csv(artifact_paths["calibration"], parse_dates=["date"])
final_forecast = pd.read_csv(artifact_paths["final_forecast"], parse_dates=["date"])
forecast_metrics = pd.read_csv(artifact_paths["forecast_metrics"])
feature_importance = pd.read_csv(artifact_paths["feature_importance"])

{name: len(table) for name, table in {
    "calibration_forecast": calibration_forecast,
    "final_forecast": final_forecast,
    "forecast_metrics": forecast_metrics,
    "feature_importance": feature_importance,
}.items()}

In [ ]:
parameter_window_audit = pd.DataFrame(
    [
        {
            "window": "model_selection",
            "expected_start": EXPECTED_WINDOWS["model_selection"][0].date().isoformat(),
            "expected_end": EXPECTED_WINDOWS["model_selection"][1].date().isoformat(),
            "artifact_start": params.get("model_selection_start", params.get("validation_start")),
            "artifact_end": params.get("model_selection_end"),
        },
        {
            "window": "policy_calibration",
            "expected_start": EXPECTED_WINDOWS["policy_calibration"][0].date().isoformat(),
            "expected_end": EXPECTED_WINDOWS["policy_calibration"][1].date().isoformat(),
            "artifact_start": params.get("policy_calibration_start"),
            "artifact_end": params.get("policy_calibration_end"),
        },
        {
            "window": "final_evaluation",
            "expected_start": EXPECTED_WINDOWS["final_evaluation"][0].date().isoformat(),
            "expected_end": EXPECTED_WINDOWS["final_evaluation"][1].date().isoformat(),
            "artifact_start": params.get("final_test_start"),
            "artifact_end": final_forecast["date"].max().date().isoformat(),
        },
    ]
)
parameter_window_audit["matches_expected"] = (
    parameter_window_audit["expected_start"].eq(parameter_window_audit["artifact_start"])
    & parameter_window_audit["expected_end"].eq(parameter_window_audit["artifact_end"])
)

observed_window_audit = pd.DataFrame(
    [
        {
            "artifact": "validation_forecast.csv",
            "purpose": "policy calibration only",
            "observed_start": calibration_forecast["date"].min(),
            "observed_end": calibration_forecast["date"].max(),
            "nonnegative": bool((calibration_forecast["forecast"] >= 0).all()),
        },
        {
            "artifact": "forecast.csv",
            "purpose": "locked final evaluation",
            "observed_start": final_forecast["date"].min(),
            "observed_end": final_forecast["date"].max(),
            "nonnegative": bool((final_forecast["forecast"] >= 0).all()),
        },
    ]
)
display(parameter_window_audit)
display(observed_window_audit)

In [ ]:
portfolio_metrics = forecast_metrics[
    (forecast_metrics["level"] == "portfolio")
    & (forecast_metrics["segment"] == "portfolio")
].copy()
class_metrics = forecast_metrics[forecast_metrics["level"] == "class"].copy()
model_mix = (
    final_forecast.groupby("model", observed=True)
    .agg(skus=("sku_id", "nunique"), rows=("forecast", "size"))
    .reset_index()
)
display(portfolio_metrics)
display(class_metrics.sort_values("segment"))
display(model_mix)
display(feature_importance.head(15))